# Scrape toner ingredients and their purpose from Paula's Choice
* Paula's Choice is a skincare brand and popular for their toners/serums -> this information is from a guide on their website
* each of these ingredients serve their own purpose and should only be used if necessary 
* toners/serums are an essentiell part in skincare -> that's why we need to save this information
* save them in a json file

In [11]:
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd

URL = "https://www.paulaschoice.de/de/beginnersguide/inhaltsstoffe"

## Scrape ingredients

In [ ]:
def scrape_paulas_choice_ingredients(url: str):
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; JuliaBot/1.0)"
    }
    resp = requests.get(url, headers=headers, timeout=10)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")

    # find all h2 headings
    h2_tags = soup.find_all("h2")

    results = []

    for h2 in h2_tags:
        heading = h2.get_text(strip=True)

        # filter newsletter/customer service
        if "Möchtest du mehr Hautpflegeberatung" in heading:
            break  
        if "Kundenservice" in heading:
            break  

        # only want headings with – (be aware that this is not the - sign)
        if "–" not in heading:
            continue

        # split topic and ingredient
        topic_part, ingredient_part = [x.strip() for x in heading.split("–", maxsplit=1)]

        # get text after heading
        text_parts = []
        for sibling in h2.next_siblings:
            if getattr(sibling, "name", None) == "h2":
                break
            if getattr(sibling, "name", None) == "p":
                para_text = sibling.get_text(" ", strip=True)
                if para_text:
                    text_parts.append(para_text)

        # sometimes multiple paragraphs
        full_text = " ".join(text_parts)

        if full_text:
            results.append(
                {
                    "topic": topic_part,
                    "ingredient": ingredient_part,
                    "text": full_text,
                    "source_url": url,
                }
            )

    return results

In [15]:
data = scrape_paulas_choice_ingredients(URL)
len(data), data

(7,
 [{'topic': 'Raue Textur',
   'ingredient': 'AHA (Alphahydroxysäure)',
   'text': 'Gibt es in vielen verschiedenen Formen. Wir empfehlen: Mandelsäure, Milchsäure, Glykolsäure. AHA ist eine nicht-abrasive Art, die Haut zu peelen, und muss nicht abgewaschen werden. Das ist viel sanfter für die Haut als jedes grobe Peeling oder jede Bürste. Wirkt direkt auf der Hautoberfläche. Ideal, wenn du auf der Suche nach einem Inhaltsstoff bist, der die Hautoberfläche glättet und geschmeidig macht.',
   'source_url': 'https://www.paulaschoice.de/de/beginnersguide/inhaltsstoffe'},
  {'topic': 'Verstopfte Poren',
   'ingredient': 'BHA (Betahydroxysäure)',
   'text': 'Du findest BHA auch unter Salicylsäure – die Form für kosmetische Produkte. Wie AHA ist auch BHA ein nicht-abrasives Peeling, das jedoch unter der Hautoberfläche wirkt und dabei hilft, Ablagerungen in den Poren zu lösen. Aus diesem Grund ist es der beste Inhaltsstoff für alle, die zu verstopften Poren neigen.',
   'source_url': 'https

## Save in json file

In [16]:
with open("paulas_choice_ingredients.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(json.dumps(data, indent=2, ensure_ascii=False))

[
  {
    "topic": "Raue Textur",
    "ingredient": "AHA (Alphahydroxysäure)",
    "text": "Gibt es in vielen verschiedenen Formen. Wir empfehlen: Mandelsäure, Milchsäure, Glykolsäure. AHA ist eine nicht-abrasive Art, die Haut zu peelen, und muss nicht abgewaschen werden. Das ist viel sanfter für die Haut als jedes grobe Peeling oder jede Bürste. Wirkt direkt auf der Hautoberfläche. Ideal, wenn du auf der Suche nach einem Inhaltsstoff bist, der die Hautoberfläche glättet und geschmeidig macht.",
    "source_url": "https://www.paulaschoice.de/de/beginnersguide/inhaltsstoffe"
  },
  {
    "topic": "Verstopfte Poren",
    "ingredient": "BHA (Betahydroxysäure)",
    "text": "Du findest BHA auch unter Salicylsäure – die Form für kosmetische Produkte. Wie AHA ist auch BHA ein nicht-abrasives Peeling, das jedoch unter der Hautoberfläche wirkt und dabei hilft, Ablagerungen in den Poren zu lösen. Aus diesem Grund ist es der beste Inhaltsstoff für alle, die zu verstopften Poren neigen.",
    "so